# Clase 14: Colas de prioridad y heaps

## Pregunta inicial

Si casi nunca necesito buscar cualquier elemento y siempre quiero consultar el menor o el mayor, ¿qué estructura debería usar?

> [!IMPORTANT]
> No respondas dentro de este notebook. Tus respuestas van en `notebook.md`, siguiendo el mismo orden.

**Responde esta pregunta en notebook.md.**


## Objetivos

Al final de la clase deberías poder:

- identificar la operación dominante de un problema;
- explicar qué es una cola de prioridad y distinguirla de una cola FIFO;
- diferenciar un heap de un BST o AVL;
- enunciar las propiedades de min-heap, max-heap y árbol binario completo;
- relacionar un heap como árbol con su representación en arreglo;
- calcular índices de padre, hijo izquierdo e hijo derecho;
- ejecutar y explicar inserción, sift-up, extracción y sift-down;
- analizar la complejidad de las operaciones principales;
- implementar y probar una clase `HeapMin`;
- resolver de forma guiada *Last Stone Weight*;
- explicar por qué una cola de prioridad prepara el camino hacia Dijkstra.


## 1. Motivación

Cada minuto llegan tareas nuevas a una agenda compartida. Cada tarea tiene un nombre y una prioridad: cuanto menor es el número, más urgente es. Después de atender una tarea debemos decidir inmediatamente cuál sigue, y mientras trabajamos pueden llegar otras todavía más urgentes.

Imagina este estado: `[(revisar servidor, 2), (responder correo, 5), (corregir examen, 3)]`. Si llega `(alarma de seguridad, 1)`, la agenda debe colocarla al frente de la atención, aunque haya llegado al final.

Podemos ensayar tres estrategias:

1. **Lista sin ordenar.** Insertar es sencillo: agregamos al final. Sin embargo, cada vez que queremos atender una tarea recorremos toda la lista para descubrir la prioridad mínima.
2. **Lista siempre ordenada.** La tarea urgente queda visible, pero insertar una tarea nueva puede desplazar muchas posiciones.
3. **Estructura especializada.** Conservamos solo el orden necesario para acceder rápidamente a la prioridad extrema, sin ordenar todos los elementos.

El hospital que atiende al paciente más urgente, el sistema operativo que elige un proceso, la simulación que ejecuta el evento más próximo y Dijkstra, que seleccionará una distancia mínima, comparten el mismo patrón: **insertar elementos y seleccionar repetidamente el de mayor prioridad**.

> [!IMPORTANT]
> Antes de elegir una estructura hay que identificar el trabajo que se repite. Optimizar una operación poco frecuente rara vez mejora el problema real.

**Pregunta adicional:** Si llegan muchas tareas mientras también atendemos tareas urgentes, ¿qué desventaja tendría mantener toda la lista siempre ordenada?

**Responde esta pregunta en notebook.md.**

Antes de elegir una estructura, debemos identificar qué operación hacemos una y otra vez.

### Comprueba tu comprensión

**Pregunta:** ¿Qué operación domina en dos de estos escenarios?

**Responde esta pregunta en notebook.md.**


## 2. Operación dominante

En la agenda no todas las acciones ocurren con la misma frecuencia. Llamamos **operación dominante** a la operación cuyo costo, por repetirse muchas veces o por ser especialmente caro, determina gran parte del tiempo total. No significa que las demás operaciones desaparezcan; significa que la elección de estructura debe responder al patrón real de uso.

Si cada minuto llega una tarea y también atendemos una, necesitamos equilibrar inserción y extracción. La siguiente tabla resume costos asintóticos típicos:

| Estrategia | Insertar | Encontrar mínimo | Extraer mínimo |
| --- | ---: | ---: | ---: |
| Lista sin ordenar | O(1) | O(n) | O(n) |
| Lista ordenada | O(n) | O(1) | O(1) |
| Heap | O(log n) | O(1) | O(log n) |

Ninguna fila gana en todo. Una lista sin ordenar es razonable si casi nunca consultamos el mínimo. Una lista ordenada puede funcionar si las consultas son abundantes y las inserciones escasas. El heap ofrece un compromiso cuando insertar y extraer se alternan continuamente.

Por ejemplo, con 1 000 tareas, una búsqueda lineal puede inspeccionar hasta 1 000 posiciones cada vez. El heap no promete ordenar las 1 000; promete mantener accesible la raíz y reparar solo un camino corto.

> [!NOTE]
> La notación O grande describe cómo crece el trabajo. No es un cronómetro exacto ni reemplaza la medición, pero permite comparar estrategias al aumentar `n`.

Ya identificamos la operación dominante; ahora necesitamos distinguir una cola ordinaria de una cola de prioridad.

### Comprueba tu comprensión

**Pregunta:** ¿Qué costo se repite si extraemos mínimos desde una lista sin ordenar?

**Responde esta pregunta en notebook.md.**


## 3. Cola FIFO vs cola de prioridad

Una cola FIFO decide por **orden de llegada**. Si llegan Ana, Bruno y Carla, el orden de atención es Ana → Bruno → Carla. La posición temporal es la regla y una llegada posterior no adelanta a las anteriores.

En la agenda aparecen estas tareas:

| Llegada | Tarea | Prioridad |
| ---: | --- | ---: |
| 1 | respaldar archivos | 4 |
| 2 | reparar servicio | 1 |
| 3 | actualizar manual | 3 |

FIFO atendería `respaldar → reparar → actualizar`. Una cola de prioridad atendería primero `reparar`, porque la clave 1 expresa mayor urgencia. Una tarea posterior puede salir antes si su prioridad lo exige. Cuando dos tareas tienen igual prioridad, podemos añadir una regla secundaria, por ejemplo conservar el orden de llegada.

La **cola de prioridad** es un tipo de dato abstracto (TDA): define operaciones como insertar, consultar el elemento prioritario y extraerlo. No obliga a usar una representación concreta. Podríamos implementarla con una lista, un árbol balanceado o un heap. En esta clase elegiremos el heap porque encaja con inserciones y extracciones repetidas.

> [!WARNING]
> “Cola de prioridad” y “heap” no son sinónimos exactos: la primera describe el contrato; el segundo es una implementación frecuente de ese contrato.

Ahora que sabemos qué comportamiento necesitamos, intentaremos descubrir manualmente una organización que lo haga posible.

### Comprueba tu comprensión

**Pregunta:** ¿Cuándo sería incorrecto usar una cola FIFO?

**Responde esta pregunta en notebook.md.**


## 4. Descubrimiento manual

Antes de formalizar el heap, diseñemos la agenda que necesitamos. Buscamos cuatro propiedades: que el mínimo esté visible, que insertar no obligue a recorrer todo, que extraer no reordene toda la colección y que la representación sea compacta.

Trabajaremos con prioridades `7, 3, 9, 1, 6, 5`. En lugar de ordenarlas por completo, colocaremos cada valor en la siguiente posición disponible y permitiremos intercambios únicamente hacia arriba cuando una prioridad nueva sea menor que su responsable inmediato.

| Valor que llega | Arreglo provisional | Comparación necesaria | Arreglo después de reparar | Intercambios |
| ---: | --- | --- | --- | ---: |
| 7 | `[7]` | no tiene padre | `[7]` | 0 |
| 3 | `[7, 3]` | `3 < 7` | `[3, 7]` | 1 |
| 9 | | | | |
| 1 | | | | |
| 6 | | | | |
| 5 | | | | |

Los dos primeros pasos muestran el método sin resolver toda la actividad. En cada fila dibuja también el árbol por niveles. Comprueba que la raíz conserva la prioridad mínima, aunque el arreglo completo no quede ordenado.

> [!TIP]
> Después de insertar, busca la única relación nueva que podría fallar: la del nodo recién agregado con su padre. No revises pares que no cambiaron.

**Pregunta adicional:** Completa la fila correspondiente a insertar `9` y explica por qué se detiene la reparación.

**Responde esta pregunta en notebook.md.**

La organización que acabamos de explorar tiene nombre y dos reglas precisas: forma completa y orden parcial.

### Comprueba tu comprensión

**Pregunta:** ¿Qué permanece estable después de insertar un valor?

**Responde esta pregunta en notebook.md.**


## 5. Qué es un heap

La organización anterior es un **heap**: un árbol binario completo que mantiene una relación de orden parcial. “Completo” describe la forma; “orden parcial” describe qué comparaciones se garantizan.

- En un **min-heap**, cada padre es menor o igual que sus hijos.
- En un **max-heap**, cada padre es mayor o igual que sus hijos.
- Un **árbol binario completo** llena cada nivel y, si el último está incompleto, coloca nodos de izquierda a derecha.

Un mismo min-heap puede verse como árbol y como arreglo:

```text
          2                 índices:  0  1  2  3  4  5  6
        /   \               valores: [2, 7, 4, 9, 8, 6, 5]
       7     4
      / \   / \
     9   8 6   5
```

La propiedad se cumple: `2 ≤ 7`, `2 ≤ 4`, `7 ≤ 9`, `7 ≤ 8`, `4 ≤ 6` y `4 ≤ 5`. Observa el contraejemplo útil: los hermanos 7 y 4 no están en orden creciente de izquierda a derecha, y aun así el heap es válido. El orden parcial basta para saber que ningún descendiente puede ser menor que la raíz.

Un BST exige otra relación: todo el subárbol izquierdo es menor que el nodo y todo el derecho es mayor. El heap no ofrece esa separación, así que buscar un valor arbitrario puede requerir revisar muchos nodos.

> [!IMPORTANT]
> El heap promete acceso eficiente al extremo de prioridad; no promete un recorrido totalmente ordenado.

Ya distinguimos forma y orden; ahora estudiaremos con cuidado el invariante del min-heap.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué un heap no es un BST?

**Responde esta pregunta en notebook.md.**


## 6. Propiedad de min-heap

El invariante central se expresa de forma local: **cada padre es menor o igual que cada uno de sus hijos existentes**. Una regla local repetida en todo el árbol produce una garantía global en la raíz.

Heap válido:

```text
        1
      /   \
     4     2
    / \   /
   9   7 6
```

Heap inválido:

```text
        1
      /   \
     4     2
    / \   /
   9   3 6   ← 3 es menor que su padre 4
```

En el árbol válido, cualquier camino que parte de la raíz nunca disminuye: `1 ≤ 4 ≤ 9`, `1 ≤ 4 ≤ 7` y `1 ≤ 2 ≤ 6`. Por eso la raíz es un mínimo global. No necesitamos comparar la raíz directamente con cada nodo; la cadena de relaciones padre-hijo transmite la garantía.

Sin embargo, la propiedad no indica en cuál subárbol está un valor arbitrario. Para buscar 6 no podemos descartar el lado izquierdo usando una regla de BST; en el peor caso habrá que inspeccionar todo el heap. Tampoco se exige relación entre hermanos: 4 puede estar a la izquierda de 2 sin invalidar el min-heap.

> [!WARNING]
> Verificar solo que la raíz sea mínima no basta. Puede existir una violación profunda aunque la raíz sea correcta.

Ya conocemos la propiedad de orden; ahora veremos por qué la forma completa es igualmente importante.

### Comprueba tu comprensión

**Pregunta:** ¿Los hermanos deben estar ordenados entre sí?

**Responde esta pregunta en notebook.md.**


## 7. Árbol binario completo

La propiedad de orden no determina dónde se ubican físicamente los nodos. Para obtener una altura pequeña y una representación compacta, el heap exige un **árbol binario completo**.

```text
Completo válido        Último nivel parcial válido     Inválido: hay un hueco
      ●                          ●                            ●
    /   \                      /   \                        /   \
   ●     ●                    ●     ●                      ●     ●
  / \   / \                  / \                          \   /
 ●   ● ●   ●                ●   ●                          ● ●
```

En el segundo dibujo, el último nivel no está lleno, pero sus nodos ocupan las posiciones disponibles de izquierda a derecha. En el tercero aparece un nodo después de un hueco: no puede representarse como una secuencia compacta sin reservar una posición vacía.

Gracias a la completitud, al recorrer el árbol por niveles obtenemos posiciones consecutivas. La raíz va en el índice 0; sus hijos en 1 y 2; el siguiente nivel en 3, 4, 5 y 6. No necesitamos guardar referencias explícitas a padre, hijo izquierdo e hijo derecho: los índices permiten calcularlas.

Además, un árbol completo con `n` nodos tiene altura proporcional a `log n`. Esa forma limita cuántos niveles pueden recorrer sift-up y sift-down.

> [!NOTE]
> “Completo” no significa que todos los nodos tengan dos hijos. El último nivel puede ser parcial; lo esencial es que no tenga huecos intermedios.

La forma ya está definida; ahora traduciremos cada posición del árbol a una celda de arreglo.

### Comprueba tu comprensión

**Pregunta:** ¿Qué ventaja da esta forma para almacenar el árbol?

**Responde esta pregunta en notebook.md.**


## 8. Representación por arreglo

No mantenemos un árbol y un arreglo por separado. El arreglo **es** el almacenamiento del heap; el dibujo del árbol solo permite interpretar sus relaciones.

Considera el heap de siete elementos `[1, 3, 2, 8, 6, 5, 4]`:

```text
                 1 (i=0)
              /           \
         3 (i=1)          2 (i=2)
        /      \          /      \
   8 (i=3)  6 (i=4)  5 (i=5)  4 (i=6)
```

| Índice | Valor | Padre | Hijo izquierdo | Hijo derecho |
| ---: | ---: | --- | --- | --- |
| 0 | 1 | — | índice 1: 3 | índice 2: 2 |
| 1 | 3 | índice 0: 1 | índice 3: 8 | índice 4: 6 |
| 2 | 2 | índice 0: 1 | índice 5: 5 | índice 6: 4 |
| 3 | 8 | índice 1: 3 | no existe | no existe |
| 4 | 6 | índice 1: 3 | no existe | no existe |
| 5 | 5 | índice 2: 2 | no existe | no existe |
| 6 | 4 | índice 2: 2 | no existe | no existe |

Cuando un valor cambia de índice durante un intercambio, también cambia de posición en el dibujo. No hay que sincronizar dos colecciones: ambas vistas se reconstruyen a partir del mismo arreglo.

> [!TIP]
> Al simular a mano, escribe siempre los índices debajo del arreglo. Reducen errores al calcular padre e hijos.

Ya dominamos la correspondencia; el siguiente paso es derivar las fórmulas que permiten navegar sin punteros.

### Comprueba tu comprensión

**Pregunta:** Para `[2, 5, 4, 9]`, ¿qué valor está en el índice 2?

**Responde esta pregunta en notebook.md.**


## 9. Fórmulas de índices

En cada nivel del árbol completo se duplica la cantidad máxima de posiciones. Con índices desde cero, los hijos de la raíz ocupan 1 y 2; los del índice 1 ocupan 3 y 4. Ese patrón produce:

```text
padre(i)     = (i - 1) // 2
izquierdo(i) = 2 * i + 1
derecho(i)   = 2 * i + 2
```

**Ejemplo resuelto: índice 1.** Su padre es `(1-1)//2 = 0`; su hijo izquierdo es `2*1+1 = 3`; su hijo derecho es `2*1+2 = 4`.

**Ejemplo guiado: índice 2.** El padre se obtiene con `(2-1)//2 = 0`. Sustituye ahora `i=2` en ambas fórmulas de hijos y comprueba el resultado contra el dibujo de siete elementos de la sección anterior.

Calcular un índice no garantiza que el nodo exista. Para un arreglo de longitud `n`, una posición es válida solo si `0 ≤ índice < n`. Un nodo puede tener hijo izquierdo y no derecho cuando el último nivel termina justo después del izquierdo. Si el izquierdo queda fuera del arreglo, el derecho también quedará fuera.

La raíz, índice 0, no tiene padre. Aplicar `(0-1)//2` en Python produciría `-1`, que apunta a la última celda y sería conceptualmente incorrecto; por eso la implementación debe tratar la raíz como caso especial.

> [!CAUTION]
> Antes de acceder a un hijo, verifica que el índice calculado sea menor que el tamaño del arreglo.

Ya podemos navegar por el heap; usaremos esas relaciones para insertar sin romper su forma.

### Comprueba tu comprensión

**Pregunta:** ¿Cuáles son los hijos del índice 2?

**Responde esta pregunta en notebook.md.**


## 10. Inserción

Insertar una nueva prioridad tiene dos obligaciones distintas:

1. conservar la forma de árbol completo;
2. restaurar la propiedad `padre ≤ hijos`.

Agregar el valor al final del arreglo resuelve automáticamente la primera. Esa celda representa la siguiente posición libre del último nivel. Solo después atendemos la segunda obligación.

**Ejemplo sin intercambio.** Partimos de `[2, 5, 7]` e insertamos 8:

```text
[2, 5, 7] → append(8) → [2, 5, 7, 8]
padre del índice 3 = índice 1; 8 ≥ 5, por lo tanto se detiene.
```

**Ejemplo con intercambio.** Partimos de `[2, 7, 9]` e insertamos 3:

```text
[2, 7, 9] → append(3) → [2, 7, 9, 3]
3 < 7 → intercambiar índices 3 y 1 → [2, 3, 9, 7]
3 ≥ 2 → detener.
```

El nuevo valor solo puede violar la relación con sus ancestros. Los demás subárboles no cambiaron y no necesitan revisarse.

**Pregunta adicional:** Si insertas `1` en `[2, 5, 4, 9, 8, 7]`, ¿con qué valores esperas que se compare antes de llegar a su posición final?

**Responde esta pregunta en notebook.md.**

Ya sabemos dónde colocar el nuevo nodo; ahora formalizaremos el movimiento ascendente que repara el orden.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué no insertamos directamente en la raíz?

**Responde esta pregunta en notebook.md.**


## 11. Sift-up

**Sift-up** significa “hacer subir” el valor recién insertado mientras sea más prioritario que su padre. No recorre todo el arreglo: sigue una sola cadena de ancestros.

Insertemos `1` en `[2, 5, 4, 9, 8, 7]`:

| Paso | Arreglo | Índice actual | Padre | Comparación | Acción |
| ---: | --- | ---: | ---: | --- | --- |
| 0 | `[2,5,4,9,8,7,1]` | 6 | 2 | `1 < 4` | intercambiar 6 ↔ 2 |
| 1 | `[2,5,1,9,8,7,4]` | 2 | 0 | `1 < 2` | intercambiar 2 ↔ 0 |
| 2 | `[1,5,2,9,8,7,4]` | 0 | — | llegó a la raíz | detener |

Después del primer intercambio, el 4 baja a una posición donde sigue siendo menor o igual que sus posibles hijos. La única duda restante acompaña al 1 hacia arriba.

```text
insertar(valor):
    agregar valor al final
    actual = último índice
    mientras actual no sea la raíz:
        padre = (actual - 1) // 2
        si arreglo[padre] <= arreglo[actual]:
            detener
        intercambiar(actual, padre)
        actual = padre
```

> [!IMPORTANT]
> El invariante se repara localmente: en cada iteración disminuye el nivel del nodo actual.

La inserción ya está completa; ahora debemos aprender a retirar la raíz sin dejar un hueco.

### Comprueba tu comprensión

**Pregunta:** ¿Cuándo se detiene sift-up?

**Responde esta pregunta en notebook.md.**


## 12. Extracción del mínimo

Consultar el mínimo solo lee la raíz. **Extraerlo** es distinto: hay que devolver ese valor y dejar una estructura válida. Si borramos directamente el índice 0, aparecería un hueco al principio y se perdería la representación compacta del árbol completo.

Usaremos el min-heap `[1, 3, 2, 8, 5]`:

1. Guardar el mínimo: `minimo = 1`.
2. Tomar el último valor, 5.
3. Eliminar la última celda y colocar 5 en la raíz: `[5, 3, 2, 8]`.
4. La forma vuelve a ser completa, pero `5 ≤ hijos` es falso.
5. Elegir el hijo menor, 2, e intercambiar: `[2, 3, 5, 8]`.
6. El 5 ya no tiene hijos; la reparación termina y se devuelve 1.

```text
Antes               Reemplazo temporal        Después de reparar
      1                    5                         2
    /   \                /   \                     /   \
   3     2              3     2                   3     5
  / \                  /                         /
 8   5                8                         8
```

Mover el último valor separa nuevamente las dos obligaciones: primero recuperamos la forma completa; después recuperamos el orden mediante sift-down.

> [!WARNING]
> No devuelvas la nueva raíz. El resultado de la operación es el mínimo que se guardó antes de modificar el arreglo.

Ya sabemos preparar la extracción; falta estudiar cómo elegir correctamente el descenso.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué se mueve el último elemento?

**Responde esta pregunta en notebook.md.**


## 13. Sift-down

**Sift-down** hace descender el valor colocado temporalmente en la raíz. En cada nivel compara con los hijos existentes, elige el menor y solo intercambia si ese hijo es menor que el nodo actual.

Supón el estado temporal `[8, 3, 2, 7, 6, 4, 5, 9]`. La raíz 8 tiene hijos 3 y 2. Si intercambiáramos con 3 por estar a la izquierda, quedaría 3 como padre de 2 y la violación continuaría. Debemos elegir 2.

| Paso | Arreglo | Actual | Hijos | Hijo menor | Acción |
| ---: | --- | ---: | --- | ---: | --- |
| 0 | `[8,3,2,7,6,4,5,9]` | 0 | 3 y 2 | 2 (i=2) | 0 ↔ 2 |
| 1 | `[2,3,8,7,6,4,5,9]` | 2 | 4 y 5 | 4 (i=5) | 2 ↔ 5 |
| 2 | `[2,3,4,7,6,8,5,9]` | 5 | no existen | — | detener |

```text
bajar(actual):
    repetir:
        menor = actual
        si existe izquierdo y arreglo[izquierdo] < arreglo[menor]: menor = izquierdo
        si existe derecho y arreglo[derecho] < arreglo[menor]: menor = derecho
        si menor == actual: detener
        intercambiar(actual, menor)
        actual = menor
```

> [!CAUTION]
> Compara el hijo derecho contra el mejor candidato actual, no únicamente contra el padre original.

Ya dominamos ambas reparaciones; ahora podremos observarlas frame por frame en el visualizador.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué debemos elegir el hijo menor?

**Responde esta pregunta en notebook.md.**


## 14. Visualizaciones ipywidgets

El visualizador reúne varias vistas del mismo estado. El panel del árbol muestra relaciones padre-hijo; el arreglo muestra los valores e índices reales; el panel textual indica comparación, decisión, acción, invariante y línea activa de pseudocódigo.

Los resaltados siempre se acompañan de texto: azul identifica el nodo actual, naranja al padre, amarillo una comparación activa, morado al hijo seleccionado, rojo un intercambio o violación y verde una posición final correcta. El color ayuda a seguir el movimiento, pero la etiqueta del índice confirma de qué nodo se trata.

Observa especialmente estas variables:

- índice y valor actuales;
- índice del padre o de ambos hijos;
- comparación booleana;
- par de índices intercambiado;
- arreglo antes y después;
- estado de la propiedad de heap.

Usa **Anterior** y **Siguiente** para analizar a tu ritmo. **Reproducir** avanza con el intervalo seleccionado y **Pausar** conserva el frame actual. En **Modo reto**, predice la decisión antes de comprobar o revelar. El slider permite saltar a un paso sin bloquear la navegación manual.

**Actividad:** selecciona “Inserción: varios intercambios”. Antes de presionar Siguiente, predice qué intercambio ocurrirá y justifícalo con índices.

**Responde esta pregunta en notebook.md.**

La primera celda de código comprueba el entorno. Si aparece un slider, ipywidgets funciona; si solo aparece texto, revisa el kernel de Jupyter o VS Code. No es un error de `HeapMin`.

Después de explorar, compararemos qué promete un heap frente a BST y AVL.

### Comprueba tu comprensión

**Pregunta:** ¿Qué relación observas entre la celda del arreglo y el nodo resaltado?

**Responde esta pregunta en notebook.md.**


In [ ]:
import ipywidgets as widgets
widgets.IntSlider(description="Prueba")


In [ ]:
from pathlib import Path
import sys
candidatas = [Path.cwd(), Path.cwd().parent, Path.cwd() / 'clase_14', Path.cwd().parent / 'clase_14']
RAIZ_CLASE = next((ruta for ruta in candidatas if (ruta / 'src' / 'visualizador_heap.py').exists()), None)
if RAIZ_CLASE is None:
    raise FileNotFoundError('Abre el notebook desde curso-alumnos o clase_14/notebooks')
sys.path.insert(0, str(RAIZ_CLASE))
from src.visualizador_heap import mostrar_insercion_heap, diagnosticar_widgets
print(diagnosticar_widgets())
mostrar_insercion_heap()


## 15. Comparación BST, AVL y heap

BST, AVL y heap son árboles binarios, pero organizan la información para contratos diferentes. Un BST separa menores a la izquierda y mayores a la derecha. Un AVL conserva esa propiedad y además controla la altura, por lo que buscar valores arbitrarios es O(log n). Un min-heap solo garantiza que cada padre sea menor o igual que sus hijos; por eso el mínimo está en la raíz, pero un valor cualquiera puede estar en muchos lugares posibles.

| Necesidad | AVL | Min-heap |
| --- | ---: | ---: |
| Buscar una clave arbitraria | O(log n) | O(n) |
| Consultar mínimo | O(log n) u O(1), según diseño | O(1) |
| Insertar | O(log n) | O(log n) |
| Extraer mínimo | O(log n) | O(log n) |
| Recorrido ordenado | sí | no |
| Tipo de orden | total por subárboles | parcial padre-hijo |

**Escenario A:** un catálogo necesita localizar productos por identificador, recorrerlos ordenadamente y consultar rangos. **Escenario B:** un simulador agrega eventos y siempre ejecuta el de menor tiempo. Elegir la misma estructura para ambos ignoraría sus operaciones dominantes.

**Pregunta adicional:** ¿Qué estructura elegirías para cada escenario y qué operación justifica tu decisión?

**Responde esta pregunta en notebook.md.**

> [!NOTE]
> Ninguna estructura es universalmente superior; una garantía adicional también tiene costo y solo es valiosa si el problema la utiliza.

Elegida la estructura, debemos justificar cuantitativamente cuánto cuesta cada operación.

### Comprueba tu comprensión

**Pregunta:** ¿Qué elegirías para búsquedas arbitrarias y qué para extraer mínimos?

**Responde esta pregunta en notebook.md.**


## 16. Complejidad

La forma completa explica los costos. Un árbol completo duplica aproximadamente su capacidad en cada nivel: 1, 2, 4, 8… Por eso almacenar `n` elementos requiere una altura O(log n).

- **Consultar el mínimo: O(1).** Se lee la celda 0; no se recorre ningún nivel.
- **Insertar: O(log n) en el peor caso.** Append es O(1) y sift-up puede recorrer todos los ancestros hasta la raíz.
- **Extraer mínimo: O(log n).** Mover el último es constante y sift-down puede descender hasta una hoja.
- **Buscar un valor arbitrario: O(n).** El orden parcial no permite descartar sistemáticamente uno de los subárboles.
- **Heapify bottom-up: O(n).** Aunque cada descenso individual puede ser logarítmico, la mayoría de los nodos está cerca de las hojas y desciende poco. Aceptaremos este resultado sin demostrar todavía la suma completa.

La inserción tiene un mejor caso O(1): el valor agregado ya es mayor o igual que su padre. Su peor caso ocurre cuando es menor que todos sus ancestros y llega hasta la raíz. De forma semejante, sift-down puede detenerse en la primera comparación o recorrer toda la altura.

> [!TIP]
> Para justificar una complejidad, nombra la estructura que se recorre: “un camino de altura logarítmica” es más informativo que memorizar `O(log n)`.

Ya conocemos el costo del mecanismo; ahora lo aplicaremos a un problema donde la prioridad máxima aparece en cada turno.

### Comprueba tu comprensión

**Pregunta:** ¿Por qué sift-up y sift-down son logarítmicos?

**Responde esta pregunta en notebook.md.**


## 17. Last Stone Weight

En *Last Stone Weight* tenemos pesos de piedras. En cada turno se eligen las dos más pesadas: si pesan lo mismo, ambas desaparecen; si pesan distinto, desaparecen y se reinserta la diferencia. El proceso termina cuando queda como máximo una piedra.

La operación dominante no es buscar un peso específico, sino **obtener repetidamente los dos máximos**. Por eso conviene una cola de prioridad máxima. Un max-heap mantiene el mayor valor en la raíz y permite extraer e insertar en O(log n).

Sigamos `[2, 7, 4, 1, 8, 1]` como multiconjunto visible. La columna “heap lógico” muestra los pesos disponibles, no necesariamente el arreglo interno exacto:

| Turno | Heap lógico antes | Dos máximos | Diferencia | Heap lógico después |
| ---: | --- | --- | ---: | --- |
| 1 | `[8,7,4,2,1,1]` | 8 y 7 | 1 | `[4,2,1,1,1]` |
| 2 | `[4,2,1,1,1]` | 4 y 2 | 2 | `[2,1,1,1]` |
| 3 | `[2,1,1,1]` | 2 y 1 | 1 | `[1,1,1]` |
| 4 | `[1,1,1]` | 1 y 1 | 0 | completa el estado final |

Python ofrece `heapq` como min-heap. Para simular prioridad máxima almacenamos cada peso negado: el peso 8 se convierte en -8, que es menor que -7 y sale primero. Al extraer, negamos de nuevo para recuperar el peso lógico.

```text
crear heap con -peso para cada piedra
mientras existan al menos dos:
    mayor = negativo de extraer_min()
    segundo = negativo de extraer_min()
    si mayor != segundo:
        insertar -(mayor - segundo)
devolver el peso restante o 0
```

Con heap, construir cuesta O(n) y cada turno realiza un número constante de operaciones O(log n): en total O(n log n). Buscar dos máximos recorriendo una lista en cada turno puede repetir O(n) muchas veces.

> [!CAUTION]
> El pseudocódigo orienta la solución, pero todavía debes implementar validación, no mutar la entrada y escribir tus propias pruebas en la carpeta de entrega.

Ya aplicamos la estructura a un problema; ahora comprobaremos sus contratos con pruebas observables.

### Comprueba tu comprensión

**Pregunta:** ¿Cuál es la operación dominante y el resultado del ejemplo?

**Responde esta pregunta en notebook.md.**


## 18. Pruebas

Una prueba útil protege una propiedad concreta. No basta con verificar un ejemplo feliz: debemos forzar las decisiones que podrían implementarse mal.

- **Raíz mínima:** después de varias inserciones, `minimo()` coincide con `min(valores)`.
- **Propiedad del arreglo:** para todo índice no raíz, el padre es menor o igual.
- **Inserción:** un valor pequeño puede recorrer varios ancestros.
- **Extracción:** el reemplazo puede descender varios niveles y elegir entre dos hijos.
- **Duplicados:** valores iguales no provocan ciclos ni se pierden.
- **Negativos:** la comparación sigue siendo numérica, sin suposiciones de positividad para `HeapMin`.
- **Operaciones consecutivas:** insertar, extraer y volver a insertar conserva tamaño e invariante.

Ejemplo completo:

```python
def test_insercion_pequena_llega_a_la_raiz():
    '''El 1 recorre varios ancestros y el heap conserva todos los valores.'''
    heap = HeapMin([2, 5, 4, 9, 8, 7])
    heap.insertar(1)
    assert heap.minimo() == 1
    assert heap.cumple_propiedad_heap()
    assert heap.tamano() == 7
```

**Pregunta adicional:** Diseña, sin escribir todavía el código completo, una entrada que obligue a `_bajar` a elegir el hijo derecho y explica qué afirmaciones comprobarías.

**Responde esta pregunta en notebook.md.**

> [!IMPORTANT]
> Una prueba debe fallar por una razón entendible si el contrato se rompe; el nombre y el comentario forman parte de esa explicación.

Con pruebas reproducibles, ya podemos revisar técnicamente el trabajo de otra persona.

### Comprueba tu comprensión

**Pregunta:** ¿Qué error específico detecta una extracción con varios descensos?

**Responde esta pregunta en notebook.md.**


## 19. Revisión técnica

Revisar no consiste en decir “está bien” ni en reescribir la solución. El revisor reproduce el comportamiento, aporta evidencia y formula mejoras accionables.

Flujo recomendado:

1. Cambia a la rama del compañero y confirma que el diff solo contiene su entrega.
2. Ejecuta las pruebas públicas con `evaluar.py`.
3. Ejecuta también `test_estudiante.py`.
4. Copia en la revisión el comando y la salida completa de `pytest -v`.
5. Revisa contrato, índices, sift-up, sift-down, casos extremos, nombres y docstrings.
6. Escribe una observación verificable: “al extraer de `[1,4,2,8,7,3,5]`, el código elige el hijo izquierdo; agrega una comparación con ambos hijos”.
7. Separa fortalezas, mejoras, conclusión y respuesta posterior del autor.

Las alertas expresan intención: `NOTE` aclara contexto, `TIP` propone estrategia, `IMPORTANT` marca un contrato esencial, `WARNING` advierte un error probable y `CAUTION` señala un riesgo serio, como ejecutar código de una rama con archivos inesperados.

**Pregunta adicional:** ¿Qué alerta usarías para indicar que el diff contiene archivos de otro alumno y por qué?

**Responde esta pregunta en notebook.md.**

> [!CAUTION]
> Antes de ejecutar una entrega, inspecciona `git status` y `git diff --name-only origin/main`.

La revisión cierra la implementación actual; ahora conectaremos la misma operación dominante con grafos ponderados.

### Comprueba tu comprensión

**Pregunta:** ¿Qué debe incluir `revision_nombre_revisor.md`?

**Responde esta pregunta en notebook.md.**


## 20. Preparación para Dijkstra

Considera un grafo con pesos no negativos:

```text
      4
  A ------- B
  |         |
1 |         | 2
  |    5    |
  C ------- D
       1 desde C a D
```

Partimos de A. Al explorar sus aristas aparecen distancias candidatas. Todavía no implementaremos el algoritmo; solo observaremos la operación que se repite.

| Momento | Distancia a A | a B | a C | a D | Menor pendiente |
| --- | ---: | ---: | ---: | ---: | --- |
| inicio | 0 | ∞ | ∞ | ∞ | A con 0 |
| después de A | 0 | 4 | 1 | ∞ | C con 1 |
| después de C | 0 | 4 | 1 | 2 | D con 2 |

La prioridad no representa el nombre del vértice, sino su **distancia tentativa**. Cada vez que aparece un camino mejor, agregamos o actualizamos una candidatura; después necesitamos seleccionar la menor distancia pendiente. Es el mismo patrón de la agenda: llegan tareas (candidaturas), atendemos la más urgente (menor distancia) y el ciclo genera nuevas tareas.

**Pregunta adicional:** Después de procesar C y descubrir una distancia 2 hacia D, ¿qué elemento debería salir antes: B con 4 o D con 2? Justifica usando la operación dominante.

**Responde esta pregunta en notebook.md.**

> [!NOTE]
> Esta sección solo motiva la cola de prioridad. La relajación completa, los vértices visitados y la implementación de Dijkstra pertenecen a la siguiente etapa.

El recorrido narrativo está completo; sintetizaremos cómo cada decisión respondió al problema inicial.

### Comprueba tu comprensión

**Pregunta:** ¿Qué guardaría la prioridad en Dijkstra?

**Responde esta pregunta en notebook.md.**


## 21. Cierre

Comenzamos con una agenda donde llegan tareas continuamente y siempre necesitamos atender la más urgente. Identificamos que la operación dominante no era buscar cualquier tarea, sino insertar y seleccionar repetidamente una prioridad extrema.

El TDA **cola de prioridad** modeló ese contrato. El **heap** proporcionó una implementación compacta: árbol binario completo almacenado en arreglo y un invariante local, `padre ≤ hijos`, que coloca el mínimo en la raíz. Insertar conserva primero la forma con append y repara el orden mediante sift-up. Extraer conserva la forma moviendo el último y repara mediante sift-down. Como ambas reparaciones recorren un solo camino de altura logarítmica, cuestan O(log n), mientras consultar la raíz cuesta O(1).

| Pregunta | Respuesta de la clase |
| --- | --- |
| ¿Qué queremos optimizar? | mínimo o máximo |
| ¿Qué TDA modela la necesidad? | cola de prioridad |
| ¿Qué implementación usamos? | heap |
| ¿Qué propiedad mantiene? | padre ≤ hijos en min-heap |
| ¿Qué veremos después? | Dijkstra |

La lección transferible no es “usar heap siempre”, sino observar el patrón de operaciones, precisar el contrato y comparar costos. Una estructura especializada puede renunciar a garantías innecesarias —como el orden total— para optimizar justo lo que el problema repite.

> [!IMPORTANT]
> Elegimos una estructura de datos según la operación que queremos optimizar.

Con esta síntesis termina la clase; conserva la pregunta final como criterio para futuros problemas.

### Comprueba tu comprensión

**Pregunta:** ¿Qué criterio usarás para elegir una estructura en un problema nuevo?

**Responde esta pregunta en notebook.md.**
